In [ ]:
# Upload the dataset to Google Colab
from google.colab import files

uploaded = files.upload()

In [ ]:
# Import the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display
from google.colab import files

In [ ]:
# Import the required libraries

# scikit-learn utilities for preprocessing and modelling (used in later stages)

# Default plot style
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

In [ ]:
# Take the name of the uploaded file
file_name = list(uploaded.keys())[0]

# Load the CSV file into a DataFrame
df = pd.read_csv(file_name)

# Display the dataset information
print(f"File successfully loaded: {file_name}")
df.info()

In [ ]:
# Display the first five rows
df.head()

# EDA


**Univariate visualisation**


In [ ]:
# Select the numeric columns to visualise
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Histogram, boxplot and KDE for every numeric column
for col in num_cols:
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

    # 1. Histogram
    sns.histplot(data=df, x=col, kde=True, color='skyblue', ax=axes[0])
    axes[0].set_title(f'Distribution of {col}')
    axes[0].set_ylabel('Frequency')

    # 2. Boxplot
    sns.boxplot(y=df[col], color='lightgreen', ax=axes[1])
    axes[1].set_title(f'Boxplot {col}')

    # 3. KDE Plot
    sns.kdeplot(data=df, x=col, fill=True, color='salmon', ax=axes[2])
    axes[2].set_title(f'Density (KDE) of {col}')
    axes[2].set_ylabel('Density')

    plt.tight_layout()
    plt.show()

In [ ]:
# Univariate visualisation of the categorical variables

# Select the categorical columns
cat_cols = df.select_dtypes(include=['object']).columns

# Gaya visual
sns.set(style="whitegrid")

# Loop over the categorical columns
for col in cat_cols:
    plt.figure(figsize=(14, 5))

    # Bar Plot (Count Plot) - versi aman tanpa warning
    plt.subplot(1, 2, 1)
    sns.countplot(y=df[col], order=df[col].value_counts().index,
                  hue=df[col], dodge=False, palette='pastel', legend=False)
    plt.title(f'Category frequency: {col}')
    plt.xlabel('Count')
    plt.ylabel(col)

    # Pie Chart
    plt.subplot(1, 2, 2)
    value_counts = df[col].value_counts()

    # Always plot the target; limit the other columns to keep the figure readable
    if col == 'Target' or len(value_counts) <= 10:
        value_counts.plot.pie(
            autopct='%1.1f%%',
            startangle=90,
            colors=sns.color_palette('pastel'),
            textprops={'fontsize': 10}
        )
        plt.title(f'Category proportion: {col}')
        plt.ylabel('')
    else:
        plt.text(0.5, 0.5, 'Too many categories for a pie chart',
                 ha='center', va='center', fontsize=12)
        plt.title(f'Category proportion: {col}')

    plt.tight_layout()
    plt.show()

**Bivariate visualisation**


In [ ]:
# Numeric vs numeric heatmap grid (computed on a copy of the DataFrame)

sns.set(style="whitegrid")

# 1. Work on a copy so the original DataFrame is not modified
df_copy = df.copy()

# 2. Select the numeric columns
numeric_cols = df_copy.select_dtypes(include=[np.number]).columns

# 3. Automatic binning helper
def create_bins(df, col, bin_size):
    """Create a binned version of a column using a fixed interval."""
    col_min, col_max = df[col].min(), df[col].max()
    bins = np.arange(col_min, col_max + bin_size, bin_size)
    labels = [f"{int(b)}-{int(b + bin_size - 1)}" for b in bins[:-1]]
    df[f"{col}_bin"] = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True)
    return df

# 4. Apply the binning to the copy
for col in numeric_cols:
    bin_size = 500 if col == "Birth Weight" else 10
    df_copy = create_bins(df_copy, col, bin_size)

# 5. Build every pair of numeric columns
combinations = [(x, y) for i, x in enumerate(numeric_cols) for y in numeric_cols[i + 1:]]

# 6. Prepare the plot grid
n_cols = 3
n_rows = int(np.ceil(len(combinations) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 5))
axes = axes.flatten()

# 7. Draw a heatmap for each pair
for i, (x, y) in enumerate(combinations):
    ax = axes[i]

    pivot_table = df_copy.pivot_table(
        index=f"{y}_bin",
        columns=f"{x}_bin",
        values=x,
        aggfunc='count',
        observed=False
    )

    sns.heatmap(
        pivot_table,
        cmap='Blues',
        linewidths=0.5,
        annot=False,
        cbar=False,
        ax=ax
    )

    ax.set_title(f"{y} vs {x}", fontsize=12)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.invert_yaxis()

# 8. Remove the unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Outlier and Missing Data


In [ ]:
# 1. Check the number of rows and columns
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# 2. Check for duplicated rows
duplicates = df[df.duplicated()]
print(f"\nNumber of duplicated rows: {duplicates.shape[0]}")

if not duplicates.empty:
    print("\nExamples of duplicated rows:")
    display(duplicates.head(5))
else:
    print("No duplicated rows found.")

# 3. Check for missing values
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

if not missing_summary.empty:
    print("\nColumns containing missing values:")
    display(missing_summary)
    print("\nPercentage of missing values (%):")
    display((df.isnull().mean() * 100).round(2))
else:
    print("\nNo missing values found.")

# Outlier Handling: Removal of Rows Containing Outliers


In [ ]:
# Columns to inspect
selected_cols = ['Waist Circumference', 'Pulmonary Function']

# Keep only the columns that exist in the DataFrame
selected_cols = [col for col in selected_cols if col in df.columns]

print(f"Boxplot of the following columns: {selected_cols}")

# Boxplot for each column
for col in selected_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[col], color='skyblue')
    plt.title(f'Outlier boxplot: {col}', fontsize=12)
    plt.xlabel(col)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

In [ ]:
# Compact outlier analysis helper
def analyse_outliers(df, col):
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    non_outliers = df[(df[col] >= lower) & (df[col] <= upper)][col]

    print(f"\nOUTLIER ANALYSIS: {col.upper()}")
    # print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
    print(f"Lower bound = {lower:.2f}")
    print(f"Upper bound = {upper:.2f}")
    print(f"Outliers = {outliers.shape[0]}, non-outliers = {non_outliers.shape[0]}")

    if len(outliers) > 0:
        print("Unique outlier values:", sorted(outliers.unique().tolist()))
    else:
        print("No outliers detected.")

    print(f"Non-outlier range: {non_outliers.min():.2f} - {non_outliers.max():.2f}")

# Run the analysis for both columns
for column in ['Waist Circumference', 'Pulmonary Function']:
    analyse_outliers(df, column)

In [ ]:
# STEP 1 - PARTITION THE RAW DATA, THEN FIT THE OUTLIER RULE ON
SEED = 42

# Test set design: TEST_PER_CLASS records per class AFTER cleaning.
TEST_PER_CLASS = 1000

# Starting candidate pool per class, before cleaning. Raised
TEST_CANDIDATE_PER_CLASS = 1400
AUTO_EXPAND = True          # False keeps the fixed pool and warns instead
MAX_ATTEMPTS = 6
SAFETY = 1.15               # extra margin on top of the measured need

# Third partition, held out from the RAW data and never cleaned.
UNSEEN_PER_CLASS = 100

TARGET_COL = 'Target'

LABELS = sorted(df[TARGET_COL].unique())

# ---- 1. the unseen partition is drawn once and never changes ----
unseen_idx = []
if UNSEEN_PER_CLASS > 0:
    for label in LABELS:
        idx = pd.Series(df.index[df[TARGET_COL] == label])
        take = min(UNSEEN_PER_CLASS, len(idx))
        unseen_idx.extend(idx.sample(n=take, random_state=SEED).tolist())

df_unseen_raw = df.loc[unseen_idx].copy() if unseen_idx else None
pool_remaining = df.drop(index=unseen_idx)

print("Raw records            :", len(df))
if df_unseen_raw is not None:
    print("Unseen partition (raw) :", len(df_unseen_raw))
print("Available for train/test:", len(pool_remaining))

# ---- helpers ----------------------------------------------------
def fit_bounds(frame, cols):
    """IQR bounds, estimated from the supplied frame only."""
    out = {}
    for col in cols:
        Q1 = frame[col].quantile(0.25)
        Q3 = frame[col].quantile(0.75)
        IQR = Q3 - Q1
        out[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return out

def drop_outliers(frame, bounds):
    """Drop rows falling outside the supplied bounds."""
    mask = pd.Series(False, index=frame.index)
    for col, (lo, hi) in bounds.items():
        if col in frame.columns:
            mask |= (frame[col] < lo) | (frame[col] > hi)
    return frame[~mask].copy(), int(mask.sum())

def drop_zero_rows(frame):
    """Rows holding a zero in any numeric column are treated as missing."""
    mask = (frame.select_dtypes(include=[np.number]) == 0).any(axis=1)
    return frame[~mask].copy(), int(mask.sum())

def draw(need_per_class):
    """Class-wise candidate draw, capped by what each class has."""
    idx = []
    for label in LABELS:
        available = pd.Series(pool_remaining.index[pool_remaining[TARGET_COL] == label])
        take = min(int(need_per_class[label]), len(available))
        idx.extend(available.sample(n=take, random_state=SEED).tolist())
    return idx

# ---- 2. size the candidate pool -------------------------------
need = {label: TEST_CANDIDATE_PER_CLASS for label in LABELS}
history = []

for attempt in range(1, MAX_ATTEMPTS + 1):
    test_idx = draw(need)
    pool_test_raw = pool_remaining.loc[test_idx].copy()
    pool_train_raw = pool_remaining.drop(index=test_idx).copy()

    OUTLIER_BOUNDS = fit_bounds(pool_train_raw, selected_cols)

    # preview the full cleaning, only to measure how many survive
    preview, _ = drop_outliers(pool_test_raw, OUTLIER_BOUNDS)
    preview, _ = drop_zero_rows(preview)
    survived = preview[TARGET_COL].value_counts()

    short = {}
    for label in LABELS:
        got = int(survived.get(label, 0))
        if got < TEST_PER_CLASS:
            rate = got / max(need[label], 1)
            room = int((pool_remaining[TARGET_COL] == label).sum())
            wanted = int(min(room, np.ceil(TEST_PER_CLASS / max(rate, 1e-6) * SAFETY)))
            if wanted > need[label]:
                short[label] = (got, wanted)

    history.append((attempt, dict(need), {k: v[0] for k, v in short.items()}))

    if not short or not AUTO_EXPAND:
        break

    print("\nAttempt {}: {} class(es) short, enlarging their candidate pool"
          .format(attempt, len(short)))
    for label, (got, wanted) in short.items():
        print("   class {:<6} survived {:>5} of {:>5} candidates -> raise to {}"
              .format(str(label), got, need[label], wanted))
        need[label] = wanted

print("\nFinal candidate pool per class:")
for label in LABELS:
    flag = "" if need[label] == TEST_CANDIDATE_PER_CLASS else "   (enlarged)"
    print("   {:<8} {:>6}{}".format(str(label), need[label], flag))

print("\nTest candidate pool    :", len(pool_test_raw))
print("Training pool          :", len(pool_train_raw))

print("\nIQR bounds fitted on the TRAINING POOL only:")
for col, (lo, hi) in OUTLIER_BOUNDS.items():
    print("  {:<24} [{:.4f}, {:.4f}]".format(col, lo, hi))

# ---- 3. apply the same bounds to both partitions ----------------
train_pool_no_out, n_tr = drop_outliers(pool_train_raw, OUTLIER_BOUNDS)
test_pool_no_out, n_te = drop_outliers(pool_test_raw, OUTLIER_BOUNDS)

print("\nOutlier rows removed:")
print("  training pool : {:>6} of {:>6}".format(n_tr, len(pool_train_raw)))
print("  test pool     : {:>6} of {:>6}   (training bounds, not its own)"
      .format(n_te, len(pool_test_raw)))

# Kept so the diagnostic cells below continue to work unchanged
df_no_outlier = pd.concat([train_pool_no_out, test_pool_no_out], ignore_index=False)
print("\nCombined records after outlier removal:", len(df_no_outlier))

# Missing Data Handling (Removal of Rows with Missing Values)


In [ ]:
# Total number of samples per diabetes class
total_per_diabetes = df_no_outlier['Target'].value_counts()

zero_by_diabetes = []

for col in numeric_cols:
    temp = df_no_outlier[df_no_outlier[col] == 0]

    if len(temp) > 0:
        counts = temp['Target'].value_counts()

        for diabetes, count in counts.items():
            total = total_per_diabetes[diabetes]
            percentage = (count / total) * 100

            zero_by_diabetes.append({
                'feature': col,
                'zero_count': count,
                'diabetes_class': diabetes,
                'percentage': percentage
            })

zero_by_diabetes = pd.DataFrame(zero_by_diabetes)

zero_by_diabetes = zero_by_diabetes.sort_values(
    ['feature', 'zero_count'],
    ascending=[True, False]
).reset_index(drop=True)

display(zero_by_diabetes)

In [ ]:
# Count the zero values in every numeric column
zero_counts = (df_no_outlier.select_dtypes(include=[np.number]) == 0).sum()

# Keep only the columns that contain zeros
zero_counts = zero_counts[zero_counts > 0]

# Display the result
print("Number of zeros per column:")
display(pd.DataFrame(zero_counts, columns=['zero_count']).sort_values('zero_count', ascending=False))

In [ ]:
# STEP 2 - TREAT ZEROS AS MISSING-VALUE INDICATORS
def drop_zero_rows(frame):
    mask = (frame.select_dtypes(include=[np.number]) == 0).any(axis=1)
    return frame[~mask].copy(), int(mask.sum())

train_pool_clean, z_tr = drop_zero_rows(train_pool_no_out)
test_pool_clean, z_te = drop_zero_rows(test_pool_no_out)

print("Rows containing a zero:")
print("  training pool : {:>6} removed, {:>6} remain".format(z_tr, len(train_pool_clean)))
print("  test pool     : {:>6} removed, {:>6} remain".format(z_te, len(test_pool_clean)))

# Kept for the diagnostic cells and the class-distribution charts
df_missing = pd.concat([train_pool_clean, test_pool_clean], ignore_index=False)

print("\nCombined clean records:", len(df_missing))

# Per-class summary
rows = []
for label in sorted(df_missing[TARGET_COL].unique()):
    rows.append([
        label,
        int((pool_train_raw[TARGET_COL] == label).sum()
            + (pool_test_raw[TARGET_COL] == label).sum()),
        int((train_pool_clean[TARGET_COL] == label).sum()),
        int((test_pool_clean[TARGET_COL] == label).sum()),
    ])

summary_table = pd.DataFrame(
    rows, columns=["Target", "Raw", "Training pool clean", "Test pool clean"]
)
print("\nPer-class record counts after cleaning:")
print(summary_table.to_string(index=False))

In [ ]:
# STEP 3 - BUILD THE BALANCED TEST SET AND THE TRAINING SET
test_final_idx = []
shortfall = []

for label in sorted(test_pool_clean[TARGET_COL].unique()):
    idx = pd.Series(test_pool_clean.index[test_pool_clean[TARGET_COL] == label])
    take = min(TEST_PER_CLASS, len(idx))
    if take < TEST_PER_CLASS:
        shortfall.append((label, take))
    test_final_idx.extend(idx.sample(n=take, random_state=SEED).tolist())

df_test = (test_pool_clean.loc[test_final_idx]
           .sample(frac=1, random_state=SEED)
           .reset_index(drop=True))

# Test-pool rows not used are returned to training rather than wasted
unused = test_pool_clean.drop(index=test_final_idx)
df_train = pd.concat([train_pool_clean, unused], ignore_index=True)

df_clean = pd.concat([df_train, df_test], ignore_index=True)

print("Training set :", len(df_train))
print("Test set     :", len(df_test))
print("Combined     :", len(df_clean))

if df_unseen_raw is not None:
    print("Unseen set   :", len(df_unseen_raw), "(raw, never cleaned)")

# ---- headroom per class ----------------------------------------
head = []
for label in sorted(pool_test_raw[TARGET_COL].unique()):
    raw_n = int((pool_test_raw[TARGET_COL] == label).sum())
    clean_n = int((test_pool_clean[TARGET_COL] == label).sum())
    head.append([label, raw_n, clean_n,
                 round(100.0 * clean_n / raw_n, 1) if raw_n else 0.0,
                 clean_n - TEST_PER_CLASS])

headroom = pd.DataFrame(
    head, columns=["Class", "Candidates", "Survived cleaning",
                   "Survival %", "Spare after test set"]
).sort_values("Spare after test set")

print("\nTest-candidate headroom, tightest class first:")
print(headroom.to_string(index=False))

tightest = headroom.iloc[0]
print("\nTightest class: {} with {} spare records."
      .format(tightest["Class"], int(tightest["Spare after test set"])))

if shortfall:
    print("\nWARNING - these classes could not supply", TEST_PER_CLASS, "test records:")
    for label, got in shortfall:
        print("   class {}: {} available".format(label, got))
    print("   Raise TEST_CANDIDATE_PER_CLASS and re-run.")
elif tightest["Spare after test set"] < 100:
    print("\nThe margin is thin. Raise TEST_CANDIDATE_PER_CLASS before")
    print("increasing UNSEEN_PER_CLASS any further.")
else:
    print("\nEvery class supplied exactly {} test records.".format(TEST_PER_CLASS))

print("\nTest-set class counts :", df_test[TARGET_COL].value_counts().sort_index().tolist())
print("Training class counts :", df_train[TARGET_COL].value_counts().sort_index().tolist())

# ---- leakage audit ---------------------------------------------
overlap = set(map(tuple, df_train.values)) & set(map(tuple, df_test.values))
print("\n--- Leakage audit ---")
print("Identical rows shared by training and test:", len(overlap))
print("IQR bounds estimated from  : training pool only")
print("Zero rule                  : parameter-free")
print("Scaler and encoders        : fitted on the training set only (cells below)")

## Partitioning, cleaning and encoding (leakage-free order)

In [ ]:
df_clean.info()

In [ ]:
# STEP 4 - SEPARATE THE FEATURES FROM THE TARGET
X_train = df_train.drop(columns=[TARGET_COL]).copy()
y_train = df_train[TARGET_COL].copy()

X_test = df_test.drop(columns=[TARGET_COL]).copy()
y_test = df_test[TARGET_COL].copy()

if df_unseen_raw is not None:
    X_unseen = df_unseen_raw.drop(columns=[TARGET_COL]).copy()
    y_unseen = df_unseen_raw[TARGET_COL].copy()
else:
    X_unseen = y_unseen = None

print("X_train:", X_train.shape, " X_test:", X_test.shape,
      " X_unseen:", None if X_unseen is None else X_unseen.shape)

## Categorical encoding


In [ ]:
# STEP 5 - CATEGORICAL ENCODING, FITTED ON THE TRAINING SET ONLY
ENCODERS = {"ordinal": {}, "binary": {}, "nominal": {}, "columns": None, "label": None}
mapping_list = []

# ---- ordinal: a fixed, externally defined order ----------------
manual_order = {
    'Physical Activity': ['Low', 'Moderate', 'High'],
    'Socioeconomic Factors': ['Low', 'Medium', 'High'],
    'Alcohol Consumption': ['Low', 'Moderate', 'High'],
}

for col, categories in manual_order.items():
    if col in X_train.columns:
        ENCODERS["ordinal"][col] = {c: i for i, c in enumerate(categories)}
        mapping_list.append(pd.DataFrame({
            'Column': col, 'Original': categories,
            'Encoded': range(len(categories)), 'Encoder': 'Manual Ordinal'}))

# ---- binary: categories taken from the training set only -------
binary_cols = [
    'Dietary Habits', 'Ethnicity', 'Liver Function Tests',
    'Genetic Markers', 'Autoantibodies', 'Family History',
    'Environmental Factors', 'Smoking Status',
    'Glucose Tolerance Test', 'History of PCOS',
    'Previous Gestational Diabetes', 'Pregnancy History',
    'Cystic Fibrosis Diagnosis', 'Steroid Use History',
    'Genetic Testing', 'Early Onset Symptoms',
]

for col in binary_cols:
    if col in X_train.columns:
        categories = list(X_train[col].dropna().unique())
        ENCODERS["binary"][col] = {c: i for i, c in enumerate(categories)}
        mapping_list.append(pd.DataFrame({
            'Column': col, 'Original': categories,
            'Encoded': range(len(categories)), 'Encoder': 'Binary Encoding'}))

# ---- nominal: one-hot, categories from the training set --------
nominal_cols = [c for c in ['Urine Test'] if c in X_train.columns]
for col in nominal_cols:
    ENCODERS["nominal"][col] = X_train[col].dropna().unique().tolist()

def encode_features(X, encoders, reference_columns=None):
    """Apply the training-set encoding to any partition."""
    out = X.copy()

    for col, mapping in encoders["ordinal"].items():
        if col in out.columns:
            out[col] = out[col].map(mapping)

    for col, mapping in encoders["binary"].items():
        if col in out.columns:
            out[col] = out[col].map(mapping)

    nominal = [c for c in encoders["nominal"] if c in out.columns]
    if nominal:
        out = pd.get_dummies(out, columns=nominal, prefix=nominal, dtype=int)

    if reference_columns is not None:
        out = out.reindex(columns=reference_columns, fill_value=0)

    return out

X_train = encode_features(X_train, ENCODERS)
ENCODERS["columns"] = X_train.columns.tolist()

X_test = encode_features(X_test, ENCODERS, ENCODERS["columns"])
if X_unseen is not None:
    X_unseen = encode_features(X_unseen, ENCODERS, ENCODERS["columns"])

for col, categories in ENCODERS["nominal"].items():
    for category in categories:
        mapping_list.append(pd.DataFrame({
            'Column': [col], 'Original': [category],
            'Encoded': ['{}_{}'.format(col, category)],
            'Encoder': ['One-Hot Encoding']}))

# ---- target ----------------------------------------------------
le_y = LabelEncoder()
y_train = le_y.fit_transform(y_train)
y_test = le_y.transform(y_test)
if y_unseen is not None:
    y_unseen = le_y.transform(y_unseen)
ENCODERS["label"] = le_y

mapping_list.append(pd.DataFrame({
    'Column': TARGET_COL + ' (Target)', 'Original': le_y.classes_,
    'Encoded': le_y.transform(le_y.classes_), 'Encoder': 'LabelEncoder'}))

print("Training features :", X_train.shape[1])
print("Test features     :", X_test.shape[1])
print("Columns identical :", X_train.columns.equals(X_test.columns))

if X_unseen is not None:
    unmapped = int(X_unseen.isna().sum().sum())
    print("Unseen features   :", X_unseen.shape[1])
    print("Unseen values with no training-set mapping:", unmapped)
    if unmapped:
        print("  These become NaN. Tree-based models accept NaN natively;")
        print("  it is the realistic cost of scoring uncleaned data.")

mapping_table = pd.concat(mapping_list, ignore_index=True)
print("\nEncoding mapping table:")
display(mapping_table)

In [ ]:
# Datasets after encoding
df_train_encoded = X_train.copy()
df_train_encoded['Target'] = y_train

df_test_encoded = X_test.copy()
df_test_encoded['Target'] = y_test

print("Training data after encoding:")
display(df_train_encoded.head())

print("\nTest data after encoding:")
display(df_test_encoded.head())

## Min-max scaling


In [ ]:
# STEP 6 - MIN-MAX SCALING, FITTED ON THE TRAINING SET ONLY
scaler = MinMaxScaler()

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Numeric features scaled:", len(num_cols))

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

if X_unseen is not None:
    X_unseen[num_cols] = scaler.transform(X_unseen[num_cols])
    below = int((X_unseen[num_cols] < 0).sum().sum())
    above = int((X_unseen[num_cols] > 1).sum().sum())
    print("\nUnseen values outside [0, 1] after scaling: {} below, {} above"
          .format(below, above))
    print("Expected: the unseen set was never cleaned, so it still holds")
    print("values beyond the training range. They are left as they are -")
    print("clipping them would hide exactly what this partition measures.")

print("\nScaler fitted on the training data only.")
display(X_train.head(3))

## Export the processed datasets


In [ ]:
# STEP 7 - EXPORT
df_train_final = X_train.copy()
df_train_final[TARGET_COL] = y_train

df_test_final = X_test.copy()
df_test_final[TARGET_COL] = y_test

df_train_final.to_csv('df_train.csv', index=False)
df_test_final.to_csv('df_test.csv', index=False)

print("Training :", df_train_final.shape, "-> df_train.csv")
print("Testing  :", df_test_final.shape, "-> df_test.csv")

if X_unseen is not None:
    df_unseen_final = X_unseen.copy()
    df_unseen_final[TARGET_COL] = y_unseen
    df_unseen_final.to_csv('df_unseen.csv', index=False)
    print("Unseen   :", df_unseen_final.shape, "-> df_unseen.csv")
else:
    df_unseen_final = None
    print("Unseen   : disabled (UNSEEN_PER_CLASS = 0)")

In [ ]:
# Summary table and bar chart of the training-test distribution

# 1. Count the samples of each target class

train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts = pd.Series(y_test).value_counts().sort_index()

# 2. Build the summary table

summary = pd.DataFrame({
    'Target': train_counts.index,
    'Training Data': train_counts.values,
    'Testing Data': test_counts.reindex(
        train_counts.index, fill_value=0
    ).values
})

summary['Total'] = summary['Training Data'] + summary['Testing Data']

summary['% Training'] = (
    summary['Training Data'] / summary['Total'] * 100
).round(2)

summary['% Testing'] = (
    summary['Testing Data'] / summary['Total'] * 100
).round(2)

# 3. Append the overall total

total_train = summary['Training Data'].sum()
total_test = summary['Testing Data'].sum()
total_all = summary['Total'].sum()

total_row = pd.DataFrame({
    'Target': ['Overall Total'],
    'Training Data': [total_train],
    'Testing Data': [total_test],
    'Total': [total_all],
    '% Training': [round(total_train / total_all * 100, 2)],
    '% Testing': [round(total_test / total_all * 100, 2)]
})

summary_final = pd.concat([summary, total_row], ignore_index=True)

# 4. Display the summary table

print("=== DATA SPLIT SUMMARY (TRAINING & TESTING) ===")

display(
    summary_final.style.format({
        'Training Data': '{:,}',
        'Testing Data': '{:,}',
        'Total': '{:,}',
        '% Training': '{:.2f}%',
        '% Testing': '{:.2f}%'
    })
)

# 5. Bar chart

plot_data = summary.copy()

labels = plot_data['Target'].astype(str)
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))

bars_train = ax.bar(
    x - width / 2,
    plot_data['Training Data'],
    width,
    label='Training Data'
)

bars_test = ax.bar(
    x + width / 2,
    plot_data['Testing Data'],
    width,
    label='Testing Data'
)

# 6. Value labels above the bars

ax.bar_label(bars_train, padding=3, fontsize=9)
ax.bar_label(bars_test, padding=3, fontsize=9)

# 7. Chart formatting

ax.set_title(
    'Distribution of Training and Testing Data per Diabetes Class',
    fontsize=14, pad=15
)

ax.set_xlabel('Diabetes Class', fontsize=11)
ax.set_ylabel('Number of Samples', fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')

ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Download the exported files

files.download('df_train.csv')
files.download('df_test.csv')

## On the unseen partition